# 🧠 exact_70 + Z3 Hybrid — Qwen3-8B LoRA-CoT + Gold FOL Z3 Entailment

**Base:** exact_70 (pure LoRA-CoT, 70.1%)
**New:** Z3 SMT solver on gold premises-FOL + Qwen Pass-2 FOL generation
**Pipeline:** Train LoRA → Parse gold FOL → Z3 entailment → Route (Z3-first, Qwen fallback)

| Component | Role |
|-----------|------|
| LoRA-CoT | Fallback: chain-of-thought reasoning |
| Gold FOL Parser | Deterministic recursive-descent on premises-FOL |
| Z3 Entailment | Check premises ⊨ hypothesis via SAT |
| Qwen Pass-2 | Base model generates FOL for question |
| Routing | Z3 definite → Z3 answer; else → Qwen-LoRA CoT |


In [ ]:
%%capture
!pip install unsloth
!pip install --upgrade --no-cache-dir trl peft accelerate bitsandbytes
!pip install datasets transformers sentencepiece protobuf
!pip install z3-solver


In [ ]:
import json
import os
import re
import random
import copy
from collections import Counter, defaultdict
from typing import List, Dict, Any, Optional, Tuple
import torch
import numpy as np
from datasets import Dataset, DatasetDict
from transformers import TrainingArguments
from trl import SFTTrainer, SFTConfig
from unsloth import FastLanguageModel
from unsloth.chat_templates import get_chat_template, train_on_responses_only

try:
    import z3
    Z3_AVAILABLE = True
    print("✅ Z3 available")
except ImportError:
    Z3_AVAILABLE = False
    print("⚠️ Z3 not available")

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print(f"✅ PyTorch {torch.__version__}, CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU: {torch.cuda.get_device_name(0)}, VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")


In [ ]:
# ============================================================
#  CONFIG
# ============================================================
DATA_PATH  = "/kaggle/input/datasets/thurdayafternoon/logic2026/Logic_Based_Educational_Queries-2.json"
OUTPUT_DIR = "/kaggle/working/lora_output_qwen3"
os.makedirs(OUTPUT_DIR, exist_ok=True)

MODEL_NAME  = "unsloth/Qwen3-8B-bnb-4bit"
MAX_SEQ_LEN = 2048

LORA_R       = 64
LORA_ALPHA   = 128
LORA_DROPOUT = 0.05
LORA_TARGET_MODULES = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"]

LEARNING_RATE    = 1e-4
BATCH_SIZE       = 2
GRAD_ACCUM_STEPS = 8
NUM_EPOCHS       = 4
WARMUP_RATIO     = 0.05
LR_SCHEDULER     = "cosine"
WEIGHT_DECAY     = 0.01
MAX_GRAD_NORM    = 1.0

TRAIN_RATIO = 0.85
VAL_RATIO   = 0.10
TEST_RATIO  = 0.05
UNKNOWN_OVERSAMPLE_FACTOR = 3

# Z3 Config
Z3_TIMEOUT_MS     = 5000
Z3_PASS2_ENABLED  = True    # Qwen base generates FOL for question
Z3_PASS2_BON      = 3       # Best-of-N for Pass-2
Z3_PASS2_TEMP     = 0.6

print("✅ Config loaded")
print(f"   Z3 Pass-2: {'ON' if Z3_PASS2_ENABLED else 'OFF'}, BoN={Z3_PASS2_BON}")


In [ ]:
with open(DATA_PATH, "r", encoding="utf-8") as f:
    raw_data: List[Dict] = json.load(f)

print(f"📦 Total records: {len(raw_data)}")
all_answers = [a for item in raw_data for a in item["answers"]]
answer_dist = Counter(all_answers)
total_ans = sum(answer_dist.values())
print(f"📊 Label distribution (total {total_ans} questions):")
for label, count in sorted(answer_dist.items()):
    print(f"  {label:10s}: {count:4d} ({count/total_ans*100:5.1f}%)")


In [ ]:
SYSTEM_PROMPT = '''You are a rigorous logical reasoning assistant specializing in First-Order Logic (FOL).
Given a set of premises (in natural language and FOL notation), you must:
1. Analyze the logical structure of each premise carefully.
2. Apply formal inference rules: modus ponens, contrapositive, universal instantiation, existential instantiation, hypothetical syllogism.
3. Reason step-by-step (Chain of Thought) BEFORE committing to a final answer.
4. For multiple-choice questions (A/B/C/D): evaluate each option against the premises.
5. For Yes/No questions: verify the statement logically.
6. If the premises are INSUFFICIENT to determine a unique conclusion, your Final Answer MUST be exactly: Unknown
7. Never hallucinate a conclusion that is not entailed by the premises.
Format your response EXACTLY as:
### Step-by-Step Reasoning
<your reasoning here>
### Final Answer
<A or B or C or D or Yes or No or Unknown>'''

def format_premises(fol_list, nl_list):
    lines = ["### Premises"]
    for i, (fol, nl) in enumerate(zip(fol_list, nl_list), 1):
        lines.append(f"P{i}. [NL]  {nl}")
        lines.append(f"   [FOL] {fol}")
    return "\n".join(lines)

def build_user_message(fol_list, nl_list, question):
    return f"{format_premises(fol_list, nl_list)}\n\n### Question\n{question}"

def build_assistant_message(explanation, answer):
    return f"### Step-by-Step Reasoning\n{explanation}\n\n### Final Answer\n{answer}"

print("✅ Prompt templates defined")


In [ ]:
def unpack_dataset(raw_data):
    samples = []
    for entry in raw_data:
        fol_list = entry["premises-FOL"]
        nl_list  = entry["premises-NL"]
        for q, a, exp in zip(entry["questions"], entry["answers"], entry["explanation"]):
            user_msg = build_user_message(fol_list, nl_list, q)
            assistant_msg = build_assistant_message(exp, a)
            samples.append({
                "messages": [
                    {"role": "system",    "content": SYSTEM_PROMPT},
                    {"role": "user",      "content": user_msg},
                    {"role": "assistant", "content": assistant_msg},
                ],
                "answer_label": a,
                "is_unknown": a.strip().lower() == "unknown",
            })
    return samples

flat_samples = unpack_dataset(raw_data)

def apply_oversampling(samples, factor=UNKNOWN_OVERSAMPLE_FACTOR):
    unknown = [s for s in samples if s["is_unknown"]]
    known   = [s for s in samples if not s["is_unknown"]]
    combined = known + unknown * factor
    random.Random(SEED).shuffle(combined)
    unk_n = sum(1 for s in combined if s["is_unknown"])
    print(f"✅ After {factor}x oversample: {len(combined)} samples, Unknown={unk_n} ({unk_n/len(combined)*100:.1f}%)")
    return combined

augmented_samples = apply_oversampling(flat_samples)


In [ ]:
def split_dataset(samples, train_r=TRAIN_RATIO, val_r=VAL_RATIO):
    shuffled = copy.deepcopy(samples)
    random.Random(SEED).shuffle(shuffled)
    n = len(shuffled)
    n_train = int(n * train_r)
    n_val   = int(n * val_r)
    def to_hf(lst): return Dataset.from_list([{"messages": s["messages"]} for s in lst])
    ds = DatasetDict({
        "train":      to_hf(shuffled[:n_train]),
        "validation": to_hf(shuffled[n_train:n_train+n_val]),
        "test":       to_hf(shuffled[n_train+n_val:]),
    })
    for name, d in ds.items(): print(f"  {name:12s}: {len(d):5d}")
    return ds

dataset = split_dataset(augmented_samples)


In [ ]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME, max_seq_length=MAX_SEQ_LEN, dtype=None, load_in_4bit=True)
tokenizer = get_chat_template(tokenizer, chat_template="qwen-2.5")

model = FastLanguageModel.get_peft_model(
    model, r=LORA_R, target_modules=LORA_TARGET_MODULES,
    lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias="none",
    use_gradient_checkpointing="unsloth", random_state=SEED)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_p   = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA: {trainable:,} trainable ({trainable/total_p*100:.2f}%)")


In [ ]:
def formatting_prompts_func(examples):
    convos = examples["messages"]
    if convos and isinstance(convos[0], dict): convos = [convos]
    return [tokenizer.apply_chat_template(c, tokenize=False, add_generation_prompt=False) for c in convos]

training_args = SFTConfig(
    output_dir=OUTPUT_DIR, packing=False, padding_free=False, assistant_only_loss=False,
    per_device_train_batch_size=BATCH_SIZE, per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM_STEPS, num_train_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE, lr_scheduler_type=LR_SCHEDULER,
    warmup_ratio=WARMUP_RATIO, weight_decay=WEIGHT_DECAY, max_grad_norm=MAX_GRAD_NORM,
    fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
    logging_steps=20, eval_strategy="steps", eval_steps=100,
    save_strategy="steps", save_steps=100, save_total_limit=3,
    load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
    seed=SEED, report_to="none", dataloader_num_workers=2, optim="adamw_8bit", push_to_hub=False)

trainer = SFTTrainer(
    model=model, processing_class=tokenizer,
    train_dataset=dataset["train"], eval_dataset=dataset["validation"],
    formatting_func=formatting_prompts_func, args=training_args)

trainer = train_on_responses_only(
    trainer, instruction_part="<|im_start|>user\n", response_part="<|im_start|>assistant\n")
print("✅ Trainer ready")


In [ ]:
print("🚀 Training...")
trainer_stats = trainer.train()
print(f"✅ Done: {trainer_stats.global_step} steps, loss={trainer_stats.metrics['train_loss']:.4f}, "
      f"time={trainer_stats.metrics['train_runtime']:.0f}s")

LORA_SAVE_PATH = os.path.join(OUTPUT_DIR, "lora_adapter")
os.makedirs(LORA_SAVE_PATH, exist_ok=True)
model.save_pretrained(LORA_SAVE_PATH)
tokenizer.save_pretrained(LORA_SAVE_PATH)
print(f"✅ LoRA saved: {LORA_SAVE_PATH}")


In [ ]:
# ============================================================
#  GOLD FOL PARSER — Recursive Descent → Z3
# ============================================================

# ── Tokenizer ──
FOL_TOKEN_SPEC = [
    ('FORALL',  r'\u2200|\\?forall\b'),
    ('EXISTS',  r'\u2203|\\?exists\b'),
    ('IFF',     r'\u2194|<->|<=>'),
    ('IMPLIES', r'\u2192|->|=>|\u2283'),
    ('AND',     r'\u2227|\u2229|/\\\\'),
    ('OR',      r'\u2228|\u222A|\\\\/'),
    ('NOT',     r'\u00AC|~'),
    ('LPAREN',  r'\('),
    ('RPAREN',  r'\)'),
    ('COMMA',   r','),
    ('IDENT',   r'[A-Za-z_][A-Za-z0-9_]*'),
    ('SKIP',    r'[ \t\n\r.]+'),
]
_fol_pattern = '|'.join(f'(?P<{n}>{p})' for n, p in FOL_TOKEN_SPEC)

def tokenize_fol(text):
    tokens = []
    for m in re.finditer(_fol_pattern, text):
        kind = m.lastgroup
        if kind == 'SKIP':
            continue
        tokens.append((kind, m.group()))
    return tokens


# ── Parser ──
class ParseError(Exception):
    pass

class FOLParser:
    # Recursive descent: FOL string -> AST tuple tree
    def __init__(self, tokens):
        self.tokens = tokens
        self.pos = 0

    def peek(self):
        return self.tokens[self.pos] if self.pos < len(self.tokens) else ('EOF', '')

    def consume(self, expected=None):
        tok = self.peek()
        if expected and tok[0] != expected:
            raise ParseError(f"Expected {expected}, got {tok}")
        self.pos += 1
        return tok

    def parse(self):
        r = self.formula()
        return r

    def formula(self):
        if self.peek()[0] in ('FORALL', 'EXISTS'):
            return self.quantifier()
        return self.biconditional()

    def quantifier(self):
        qtype = self.consume()[0]
        variables = []
        while self.peek()[0] == 'IDENT':
            name = self.peek()[1]
            if len(name) <= 2 and name[0].islower():
                variables.append(self.consume()[1])
            else:
                break
            if self.peek()[0] in ('FORALL', 'EXISTS'):
                inner = self.quantifier()
                result = inner
                for v in reversed(variables):
                    result = (qtype, v, result)
                return result

        if not variables:
            raise ParseError(f"Quantifier without variable at pos {self.pos}")

        if self.peek()[0] == 'LPAREN':
            self.consume('LPAREN')
            body = self.formula()
            if self.peek()[0] == 'RPAREN':
                self.consume('RPAREN')
        else:
            body = self.formula()

        result = body
        for v in reversed(variables):
            result = (qtype, v, result)
        return result

    def biconditional(self):
        left = self.implication()
        while self.peek()[0] == 'IFF':
            self.consume()
            left = ('IFF', left, self.implication())
        return left

    def implication(self):
        left = self.disjunction()
        if self.peek()[0] == 'IMPLIES':
            self.consume()
            return ('IMPLIES', left, self.implication())
        return left

    def disjunction(self):
        left = self.conjunction()
        while self.peek()[0] == 'OR':
            self.consume()
            left = ('OR', left, self.conjunction())
        return left

    def conjunction(self):
        left = self.unary()
        while self.peek()[0] == 'AND':
            self.consume()
            left = ('AND', left, self.unary())
        return left

    def unary(self):
        if self.peek()[0] == 'NOT':
            self.consume()
            return ('NOT', self.unary())
        if self.peek()[0] in ('FORALL', 'EXISTS'):
            return self.quantifier()
        return self.atom()

    def atom(self):
        if self.peek()[0] == 'LPAREN':
            self.consume('LPAREN')
            r = self.formula()
            if self.peek()[0] == 'RPAREN':
                self.consume('RPAREN')
            return r
        if self.peek()[0] == 'IDENT':
            name = self.consume()[1]
            if self.peek()[0] == 'LPAREN':
                self.consume('LPAREN')
                args = [self.term()]
                while self.peek()[0] == 'COMMA':
                    self.consume()
                    args.append(self.term())
                if self.peek()[0] == 'RPAREN':
                    self.consume('RPAREN')
                return ('PRED', name, args)
            return ('ATOM', name)
        raise ParseError(f"Unexpected: {self.peek()}")

    def term(self):
        if self.peek()[0] == 'IDENT':
            return self.consume()[1]
        raise ParseError(f"Expected term, got {self.peek()}")


# ── Z3 Compiler ──
class Z3Compiler:
    # Compile AST to Z3 expressions. Shared across premises.
    def __init__(self):
        self.sort = z3.DeclareSort('E')
        self._funcs = {}
        self._consts = {}
        self._props = {}

    def get_func(self, name, arity):
        key = (name, arity)
        if key not in self._funcs:
            self._funcs[key] = z3.Function(name, *([self.sort]*arity), z3.BoolSort())
        return self._funcs[key]

    def get_const(self, name):
        if name not in self._consts:
            self._consts[name] = z3.Const(name, self.sort)
        return self._consts[name]

    def get_prop(self, name):
        if name not in self._props:
            self._props[name] = z3.Bool(name)
        return self._props[name]

    def compile(self, ast, bound=None):
        if bound is None:
            bound = {}
        tag = ast[0]

        if tag == 'FORALL':
            _, var, body = ast
            v = z3.Const(var, self.sort)
            return z3.ForAll([v], self.compile(body, {**bound, var: v}))

        elif tag == 'EXISTS':
            _, var, body = ast
            v = z3.Const(var, self.sort)
            return z3.Exists([v], self.compile(body, {**bound, var: v}))

        elif tag == 'IMPLIES':
            return z3.Implies(self.compile(ast[1], bound), self.compile(ast[2], bound))

        elif tag == 'AND':
            return z3.And(self.compile(ast[1], bound), self.compile(ast[2], bound))

        elif tag == 'OR':
            return z3.Or(self.compile(ast[1], bound), self.compile(ast[2], bound))

        elif tag == 'IFF':
            l = self.compile(ast[1], bound)
            r = self.compile(ast[2], bound)
            return l == r

        elif tag == 'NOT':
            return z3.Not(self.compile(ast[1], bound))

        elif tag == 'PRED':
            _, name, args = ast
            z3_args = [bound[a] if a in bound else self.get_const(a) for a in args]
            return self.get_func(name, len(args))(*z3_args)

        elif tag == 'ATOM':
            _, name = ast
            if name in bound:
                raise ParseError(f"Variable {name} used as proposition")
            return self.get_prop(name)

        raise ParseError(f"Unknown AST: {tag}")


# ── Convenience functions ──
def parse_fol_to_z3(fol_str, compiler, bound=None):
    # Parse a single FOL string and compile to Z3. Returns None on failure.
    try:
        tokens = tokenize_fol(fol_str)
        if not tokens:
            return None
        parser = FOLParser(tokens)
        ast = parser.parse()
        return compiler.compile(ast, bound or {})
    except Exception:
        return None


def parse_all_premises(fol_list):
    # Parse all gold premises-FOL. Returns (z3_exprs, compiler, n_parsed, n_total).
    compiler = Z3Compiler()
    z3_exprs = []
    n_ok = 0
    for fol_str in fol_list:
        expr = parse_fol_to_z3(fol_str, compiler)
        if expr is not None:
            z3_exprs.append(expr)
            n_ok += 1
    return z3_exprs, compiler, n_ok, len(fol_list)


# ── Test ──
if Z3_AVAILABLE:
    _test_cases = [
        "\u2200x (P(x) \u2192 Q(x))",
        "\u2203x (R(x) \u2227 S(x))",
        "\u00ACP(a)",
        "\u2200x\u2200y (T(x,y) \u2192 T(y,x))",
        "P(a) \u2192 (Q(b) \u2227 R(c))",
        "\u2200x (A(x) \u2194 B(x))",
    ]
    _comp = Z3Compiler()
    _ok = 0
    for tc in _test_cases:
        r = parse_fol_to_z3(tc, _comp)
        status = "\u2705" if r is not None else "\u274C"
        if r is not None:
            _ok += 1
        print(f"  {status} {tc:50s} \u2192 {str(r)[:60] if r else 'FAIL'}")
    print(f"\n\u2705 FOL Parser: {_ok}/{len(_test_cases)} passed")


In [ ]:
# ============================================================
#  Z3 ENTAILMENT ENGINE
# ============================================================

def z3_check_entailment(premises_z3, hypothesis_z3, timeout_ms=Z3_TIMEOUT_MS):
    """
    Check: premises |= hypothesis?
    1. SAT(premises & ~H) -> UNSAT -> 'Yes'
    2. SAT(premises & H)  -> UNSAT -> 'No'
    3. Both SAT -> 'Unknown'
    """
    if not Z3_AVAILABLE or not premises_z3:
        return None, "no_z3"
    try:
        solver = z3.Solver()
        solver.set("timeout", timeout_ms)
        for p in premises_z3:
            solver.add(p)

        # Check premises & ~H
        solver.push()
        solver.add(z3.Not(hypothesis_z3))
        res_neg = solver.check()
        solver.pop()

        if res_neg == z3.unsat:
            return 'Yes', 'definite'

        # Check premises & H
        solver.push()
        solver.add(hypothesis_z3)
        res_pos = solver.check()
        solver.pop()

        if res_pos == z3.unsat:
            return 'No', 'definite'

        if res_neg == z3.sat and res_pos == z3.sat:
            return 'Unknown', 'definite'

        return None, 'timeout'
    except Exception as e:
        return None, f'error:{e}'


def z3_check_mcq(premises_z3, options_z3, timeout_ms=Z3_TIMEOUT_MS):
    """For MCQ: check which option is uniquely entailed."""
    if not Z3_AVAILABLE or not premises_z3:
        return None, {}
    try:
        solver = z3.Solver()
        solver.set("timeout", timeout_ms)
        for p in premises_z3:
            solver.add(p)
        results = {}
        entailed = []
        for label, opt_z3 in sorted(options_z3.items()):
            solver.push()
            solver.add(z3.Not(opt_z3))
            res = solver.check()
            solver.pop()
            if res == z3.unsat:
                results[label] = 'entailed'
                entailed.append(label)
            elif res == z3.sat:
                results[label] = 'not_entailed'
            else:
                results[label] = 'unknown'
        if len(entailed) == 1:
            return entailed[0], results
        return None, results
    except Exception:
        return None, {}


# ── Question FOL extraction ──
def extract_fol_from_question(question):
    """Try to extract FOL formula embedded in question text."""
    fol_symbols = set('\u2200\u2203\u2192\u2227\u2228\u00AC\u2194')

    # Pattern: Statement: <FOL>
    m = re.search(r"Statement\s*:\s*['\"]?(.+?)['\"]?\s*$", question, re.DOTALL | re.MULTILINE)
    if m:
        stmt = m.group(1).strip().rstrip("'\"")
        if any(c in stmt for c in fol_symbols):
            return stmt
        if re.search(r'[A-Z][a-zA-Z_]*\([^)]+\)', stmt):
            return stmt

    # Direct FOL anywhere in question
    for line in question.split('\n'):
        line = line.strip()
        if any(c in line for c in fol_symbols) and len(line) > 3:
            if not re.match(r'^[A-D][.)]\s', line):
                return line
    return None


def extract_mcq_options(question):
    """Extract MCQ options. Returns dict {A: text, ...} or None."""
    options = {}
    for label in ['A', 'B', 'C', 'D']:
        pattern = rf'(?:^|\n)\s*{label}[.)]\s*(.+?)(?=\n\s*[B-D][.)]\s|\Z)'
        m = re.search(pattern, question, re.DOTALL)
        if m:
            options[label] = m.group(1).strip()
    return options if len(options) >= 2 else None


def classify_question(question):
    """Classify: 'yesno' or 'mcq'."""
    q_lower = question.lower()
    if any(x in q_lower for x in ['(a)', '(b)', 'option a', 'option b']):
        return 'mcq'
    if 'which of the following' in q_lower:
        return 'mcq'
    for line in question.split('\n'):
        if re.match(r'\s*[A-D][.)]\s', line):
            return 'mcq'
    return 'yesno'


# ── Test ──
if Z3_AVAILABLE:
    _comp = Z3Compiler()
    _p1 = parse_fol_to_z3("\u2200x (P(x) \u2192 Q(x))", _comp)
    _p2 = parse_fol_to_z3("P(a)", _comp)
    _h = parse_fol_to_z3("Q(a)", _comp)
    ans, conf = z3_check_entailment([_p1, _p2], _h)
    print(f"\u2705 Entailment: \u2200x(P\u2192Q), P(a) |= Q(a)? -> {ans} ({conf})")
    assert ans == 'Yes'

    _h2 = parse_fol_to_z3("\u00ACQ(a)", _comp)
    ans2, _ = z3_check_entailment([_p1, _p2], _h2)
    print(f"\u2705 Entailment: \u2200x(P\u2192Q), P(a) |= \u00ACQ(a)? -> {ans2}")
    assert ans2 == 'No'
    print("\u2705 Z3 Entailment Engine ready")


In [ ]:
# ============================================================
#  PASS-2: Qwen base -> FOL for question hypothesis
# ============================================================

PASS2_SYSTEM = (
    "You are a First-Order Logic (FOL) translator.\n"
    "Given premises in FOL and a question/statement, output ONLY the FOL formula "
    "for the hypothesis being tested.\n"
    "Use the SAME predicate names and constant names as in the premises.\n"
    "Output ONLY the FOL formula on a single line. No explanation, no markdown, no prefix.\n"
    "Example output: \u2200x (P(x) \u2192 Q(x))"
)

def build_pass2_prompt(fol_list, nl_list, question):
    prem = format_premises(fol_list, nl_list)
    return (
        f"{prem}\n\n"
        f"### Question\n{question}\n\n"
        "### Task\n"
        "Write the FOL formula for the hypothesis or statement being tested.\n"
        "Use EXACTLY the same predicate and constant names from the premises.\n"
        "Output ONLY the FOL formula, nothing else."
    )


def qwen_generate_fol(model, tokenizer, fol_list, nl_list, question,
                       n_samples=Z3_PASS2_BON, temperature=Z3_PASS2_TEMP):
    """Use Qwen BASE model (disable_adapter) to generate FOL for the question."""
    messages = [
        {"role": "system", "content": PASS2_SYSTEM},
        {"role": "user",   "content": build_pass2_prompt(fol_list, nl_list, question)},
    ]
    try:
        input_ids = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_tensors="pt", enable_thinking=False,
        ).to(model.device)
    except TypeError:
        input_ids = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_tensors="pt",
        ).to(model.device)

    results = []
    for _ in range(n_samples):
        with model.disable_adapter():
            with torch.no_grad():
                out = model.generate(
                    input_ids,
                    max_new_tokens=128,
                    max_length=None,
                    temperature=max(temperature, 0.01),
                    do_sample=True,
                    repetition_penalty=1.0,
                    pad_token_id=tokenizer.eos_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
        new_tokens = out[0][input_ids.shape[1]:]
        text = tokenizer.decode(new_tokens, skip_special_tokens=True).strip()
        text = re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()
        # take first non-empty line
        for line in text.split('\n'):
            line = line.strip()
            if line and not line.startswith('#'):
                results.append(line)
                break
    return results

print("\u2705 Pass-2 FOL generation ready")


In [ ]:
# ============================================================
#  INFERENCE - Qwen-LoRA CoT + Z3 Hybrid
# ============================================================
FastLanguageModel.for_inference(model)

def extract_final_answer(text):
    text = text.strip()
    m = re.search(r'###\s*Final\s*Answer[\s:]*\n?\s*(.+)', text, re.IGNORECASE)
    if m:
        ans = m.group(1).strip().split('\n')[0].strip()
        if re.match(r'^unknown', ans, re.IGNORECASE): return 'Unknown'
        ml = re.match(r'^([A-D])[.)\s:]', ans, re.IGNORECASE)
        if ml: return ml.group(1).upper()
        if re.match(r'^[A-D]$', ans, re.IGNORECASE): return ans.upper()
        if re.match(r'^yes', ans, re.IGNORECASE): return 'Yes'
        if re.match(r'^no', ans, re.IGNORECASE): return 'No'
    m2 = re.search(r'(?:answer\s*(?:is|:)\s*)([A-D]|unknown|yes|no)', text, re.IGNORECASE)
    if m2:
        g = m2.group(1).strip().lower()
        if g == 'unknown': return 'Unknown'
        if g == 'yes': return 'Yes'
        if g == 'no': return 'No'
        return g.upper()
    last = text[-500:]
    m3 = re.findall(r'\b([A-D])\b', last)
    if m3: return m3[-1].upper()
    if re.search(r'\bunknown\b', text, re.IGNORECASE): return 'Unknown'
    if re.search(r'\byes\b', text, re.IGNORECASE): return 'Yes'
    if re.search(r'\bno\b', text, re.IGNORECASE): return 'No'
    return 'UNPARSEABLE'


def predict_qwen(model, tokenizer, fol_list, nl_list, question,
                  max_new_tokens=256, temperature=0.0):
    """Qwen-LoRA CoT prediction (same as exact_70)."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": build_user_message(fol_list, nl_list, question)},
    ]
    try:
        input_ids = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_tensors="pt", enable_thinking=False,
        ).to(model.device)
    except TypeError:
        input_ids = tokenizer.apply_chat_template(
            messages, tokenize=True, add_generation_prompt=True,
            return_tensors="pt",
        ).to(model.device)
    with torch.no_grad():
        out = model.generate(
            input_ids, max_new_tokens=max_new_tokens, max_length=None,
            temperature=temperature, do_sample=temperature > 0,
            repetition_penalty=1.0, pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id)
    new_tokens = out[0][input_ids.shape[1]:]
    raw = tokenizer.decode(new_tokens, skip_special_tokens=True)
    raw = re.sub(r'<think>.*?</think>', '', raw, flags=re.DOTALL).strip()
    return extract_final_answer(raw), raw


def hybrid_predict(model, tokenizer, entry, question):
    """
    Z3-first hybrid:
    1. Parse gold premises-FOL -> Z3
    2. Try embedded FOL from question -> Z3 entailment
    3. Try Qwen Pass-2 FOL -> Z3 entailment
    4. Fallback: Qwen-LoRA CoT
    """
    fol_list = entry["premises-FOL"]
    nl_list  = entry["premises-NL"]

    # Stage 1: Parse gold premises
    premises_z3, compiler, n_ok, n_total = parse_all_premises(fol_list)
    if n_ok < max(1, n_total * 0.3):
        ans, raw = predict_qwen(model, tokenizer, fol_list, nl_list, question)
        return ans, 'qwen_only(parse_fail)', {}

    qtype = classify_question(question)

    # Stage 2: Try embedded FOL from question
    if qtype == 'yesno':
        fol_str = extract_fol_from_question(question)
        if fol_str:
            hyp_z3 = parse_fol_to_z3(fol_str, compiler)
            if hyp_z3 is not None:
                z3_ans, z3_conf = z3_check_entailment(premises_z3, hyp_z3)
                if z3_ans is not None and z3_conf == 'definite':
                    return z3_ans, 'z3_embedded', {}

    # Stage 3: Qwen Pass-2 -> Z3 (Yes/No)
    if Z3_PASS2_ENABLED and qtype == 'yesno':
        fol_candidates = qwen_generate_fol(
            model, tokenizer, fol_list, nl_list, question)
        z3_votes = []
        for fc in fol_candidates:
            hyp_z3 = parse_fol_to_z3(fc, compiler)
            if hyp_z3 is not None:
                z3_ans, z3_conf = z3_check_entailment(premises_z3, hyp_z3)
                if z3_ans is not None and z3_conf == 'definite':
                    z3_votes.append(z3_ans)
        if z3_votes:
            vote_counts = Counter(z3_votes)
            best, best_n = vote_counts.most_common(1)[0]
            if best_n >= 2 or len(z3_votes) == 1:
                return best, 'z3_pass2', {'votes': dict(vote_counts)}

    # Stage 3b: MCQ Pass-2
    if Z3_PASS2_ENABLED and qtype == 'mcq':
        mcq_opts_text = extract_mcq_options(question)
        if mcq_opts_text:
            opts_z3 = {}
            for label, opt_text in mcq_opts_text.items():
                fol_cands = qwen_generate_fol(
                    model, tokenizer, fol_list, nl_list,
                    f"Translate to FOL: {opt_text}", n_samples=1, temperature=0.3)
                if fol_cands:
                    opt_z3 = parse_fol_to_z3(fol_cands[0], compiler)
                    if opt_z3 is not None:
                        opts_z3[label] = opt_z3
            if len(opts_z3) >= 2:
                mcq_ans, mcq_res = z3_check_mcq(premises_z3, opts_z3)
                if mcq_ans is not None:
                    return mcq_ans, 'z3_mcq', {'results': mcq_res}

    # Stage 4: Fallback Qwen-LoRA CoT
    ans, raw = predict_qwen(model, tokenizer, fol_list, nl_list, question)
    return ans, 'qwen_fallback', {}


print("\u2705 Hybrid predict ready")
print("   Routes: z3_embedded -> z3_pass2 -> z3_mcq -> qwen_fallback")


In [ ]:
# ============================================================
#  EVALUATION
# ============================================================
print("📊 Evaluating hybrid pipeline...")
print("=" * 70)

correct = 0
total_eval = 0
results_log = []
route_counter = Counter()
route_correct = defaultdict(int)
route_total = defaultdict(int)

for idx_i, entry in enumerate(raw_data):
    for q_i, (q, expected_ans, _) in enumerate(
        zip(entry["questions"], entry["answers"], entry["explanation"])
    ):
        pred, route, details = hybrid_predict(model, tokenizer, entry, q)
        is_correct = (pred.strip().lower() == expected_ans.strip().lower())

        total_eval += 1
        if is_correct: correct += 1
        route_counter[route] += 1
        route_total[route] += 1
        if is_correct: route_correct[route] += 1

        results_log.append({
            "idx": idx_i, "q_i": q_i, "expected": expected_ans,
            "pred": pred, "correct": is_correct, "route": route,
        })

        if (idx_i + 1) % 50 == 0 and q_i == 0:
            acc = correct / total_eval * 100
            print(f"  [{idx_i+1:3d}/{len(raw_data)}] Acc: {acc:.1f}%  Routes: {dict(route_counter)}")

print("\n" + "=" * 70)
print("✅ FINAL RESULTS")
print("=" * 70)
overall_acc = correct / total_eval * 100
print(f"Overall Accuracy: {correct}/{total_eval} = {overall_acc:.1f}%")

print("\n--- Accuracy by Route ---")
for route in sorted(route_total.keys()):
    n = route_total[route]
    c = route_correct[route]
    pct = c/n*100 if n else 0
    share = n/total_eval*100
    print(f"  {route:25s}: {c:4d}/{n:4d} = {pct:5.1f}% acc  ({share:5.1f}% of questions)")

print("\n--- Per-class accuracy ---")
class_correct = defaultdict(int)
class_total = defaultdict(int)
for r in results_log:
    class_total[r["expected"]] += 1
    if r["correct"]: class_correct[r["expected"]] += 1
for label in sorted(class_total.keys()):
    n = class_total[label]
    c = class_correct[label]
    print(f"  {label:10s}: {c:4d}/{n:4d} = {c/n*100:5.1f}%")

print("\n--- Confusion (expected -> predicted) ---")
confusion = defaultdict(Counter)
for r in results_log: confusion[r["expected"]][r["pred"]] += 1
for exp in sorted(confusion.keys()):
    row = confusion[exp]
    parts = " ".join(f"{p}={c}" for p, c in sorted(row.items()))
    print(f"  {exp:10s} (n={sum(row.values()):3d}): {parts}")

print(f"\n{'\U0001f3af TARGET MET' if overall_acc >= 70 else '\u26a0\ufe0f Below target'}: {overall_acc:.1f}%")


In [ ]:
wrong = [r for r in results_log if not r["correct"]]
print(f"\n🔍 {len(wrong)} wrong predictions")
print("\n--- 5 wrong samples ---")
for r in wrong[:5]:
    entry = raw_data[r["idx"]]
    q = entry["questions"][r["q_i"]]
    print(f"  [{r['idx']},{r['q_i']}] route={r['route']}")
    print(f"    Q: {q[:80]}...")
    print(f"    Expected: {r['expected']} | Predicted: {r['pred']}")
    print()
